In [ ]:
#import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print("Libraries loaded successfully")


In [ ]:
import pandas as pd
import os

try:
    from google.colab import drive
    os.system("git clone https://github.com/J4m331/COMP3608-IS-Project")
    BASE_DIR = "/content/COMP3608-IS-Project"
except ImportError:
    BASE_DIR = os.path.dirname(os.path.abspath("GA.ipynb"))

df1 = pd.read_csv(os.path.join(BASE_DIR, "data", "vgData2024.csv"))
df2 = pd.read_csv(os.path.join(BASE_DIR, "data", "vgData2026.csv"))

print(f"df1: {df1.shape}")
print(f"df2: {df2.shape}")

print("All datasets loaded!")

In [ ]:
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())

In [ ]:

#Fix Data Types
#Sales columns loaded as text - convert to numbers

# df1 and df2 sales columns
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']

for col in sales_cols:
    df1[col] = pd.to_numeric(df1[col], errors='coerce')
    df2[col] = pd.to_numeric(df2[col], errors='coerce')

# df1 and df2 year column
df1['Release_Year'] = pd.to_numeric(df1['Release_Year'], errors='coerce')
df2['Release_Year'] = pd.to_numeric(df2['Release_Year'], errors='coerce')



print("\ndf1 dtypes:\n", df1.dtypes)
print("\ndf2 dtypes:\n", df2.dtypes)


print("Data types fixed")



In [ ]:
#Creating a Global_sales for df1 and df2


#Formula: Global_Sales = NA + EU + JP + Other

df1['Global_Sales'] = (df1['NA_Sales'] + df1['EU_Sales'] +
                        df1['JP_Sales'] + df1['Other_Sales'])

df2['Global_Sales'] = (df2['NA_Sales'] + df2['EU_Sales'] +
                        df2['JP_Sales'] + df2['Other_Sales'])

print("Global_Sales created for df1 and df2")
print(f"\ndf1 Global_Sales → Min: {df1['Global_Sales'].min():.2f} | "
      f"Max: {df1['Global_Sales'].max():.2f} | "
      f"Mean: {df1['Global_Sales'].mean():.2f}")

print(f"df2 Global_Sales → Min: {df2['Global_Sales'].min():.2f} | "
      f"Max: {df2['Global_Sales'].max():.2f} | "
      f"Mean: {df2['Global_Sales'].mean():.2f}")




In [ ]:
# Global Sales Distribution
# Choosing Low / Medium / High thresholds
# Percentile table
print(" Global_Sales Percentiles (in millions):")
print(f"\n{'Dataset':<10} {'25th':>8} {'50th':>8} {'75th':>8} {'90th':>8} {'Max':>8}")
print("-" * 50)

for name, df in [("df1", df1), ("df2", df2)]:
    p = df['Global_Sales'].quantile([0.25, 0.5, 0.75, 0.9])
    mx = df['Global_Sales'].max()

    print(f"{name:<10} {p[0.25]:>8.2f} {p[0.50]:>8.2f} "
          f"{p[0.75]:>8.2f} {p[0.90]:>8.2f} {mx:>8.2f}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, (name, df) in zip(axes, [("df1 (vgData2024)", df1),
                                   ("df2 (vgData2026)", df2)]):
    ax.hist(df['Global_Sales'], bins=50,
            color='steelblue', edgecolor='black')
    ax.set_title(f'{name}\nGlobal Sales Distribution')
    ax.set_xlabel('Global Sales (millions)')
    ax.set_ylabel('Number of Games')
    ax.set_xlim(0, 5)

plt.suptitle('Global Sales Distribution Across Datasets',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
##Creating the Profitability Target Variable
# Low/ Medium / High based on Global_Sales
# The threshold is based on data distrubution
#   Low    = below median       (< 0.20M)
#   Medium = middle range       (0.20M - 1.00M)
#   High   = top sellers        (> 1.00M)


def create_profitability(df):
    conditions = [
        df['Global_Sales'] < 0.20,
        (df['Global_Sales'] >= 0.20) & (df['Global_Sales'] <= 1.00),
        df['Global_Sales'] > 1.00
    ]
    labels = ['Low', 'Medium', 'High']
    df['Profitability'] = np.select(conditions, labels, default='Low')
    return df

df1 = create_profitability(df1)
df2 = create_profitability(df2)


# Check class distribution for each dataset
print("Class Distribution:")
for name, df in [("df1", df1), ("df2", df2)]:
    counts = df['Profitability'].value_counts()
    total = len(df)
    print(f"\n{name}:")
    for label in ['Low', 'Medium', 'High']:
        count = counts.get(label, 0)
        pct = (count / total) * 100
        print(f"  {label:<8}: {count:>6} games ({pct:.1f}%)")



In [ ]:
# Feature Selection
# We will choose which columns to use as features (X)

# df1 features
df1_features = ['Platform', 'Genre', 'Publisher',
                 'Release_Year', 'NA_Sales', 'EU_Sales',
                 'JP_Sales', 'Other_Sales']


# df2 features (same structure as df1)
df2_features = ['Platform', 'Genre', 'Publisher',
                 'Release_Year', 'NA_Sales', 'EU_Sales',
                 'JP_Sales', 'Other_Sales']

print("Features selected")
print(f"\ndf1 features: {df1_features}")
print(f"df2 features: {df2_features}")


In [ ]:
# Encoding Categorical Variables
# This is done because Random Forest cannot read text, therefore it must be converted to numbers
# We use Label Encoding for : Platform, Genre, Publisher

from sklearn.preprocessing import LabelEncoder

def encode_dataframe(df, feature_cols):
    df_encoded = df[feature_cols + ['Profitability']].copy()

    # Encode categorical columns
    cat_cols = ['Platform', 'Genre', 'Publisher']
    le = LabelEncoder()

    for col in cat_cols:
        if col in df_encoded.columns:
            df_encoded[col] = le.fit_transform(
                df_encoded[col].astype(str))

    return df_encoded

df1_encoded = encode_dataframe(df1, df1_features)
df2_encoded = encode_dataframe(df2, df2_features)


print("Encoding complete")
print("\n df1 sample after encoding:")
print(df1_encoded.head(3))
print("\n df2 sample after encoding:")
print(df2_encoded.head(3))


In [ ]:
# Split Features and Target
# X = all Feature columns
# Y = PRofitiability (Low / Medium / High )

# df1
X1 = df1_encoded[df1_features]
y1 = df1_encoded['Profitability']

# df2
X2 = df2_encoded[df2_features]
y2 = df2_encoded['Profitability']




print("Features and target split complete")
print(f"\ndf1 → X1: {X1.shape} | y1: {y1.shape}")
print(f"df2 → X2: {X2.shape} | y2: {y2.shape}")


print(f"\nTarget classes: {sorted(y1.unique())}")




In [ ]:
# Training / Test Split
# 80 % training, 20% testing
# random_state= 42 ensures reproducible results
# stratify=y keeps class proportions in both splits


from sklearn.model_selection import train_test_split

# df1 split
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1)

# df2 split
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)



print("Train/Test splits complete")
print(f"\ndf1 → Train: {X1_train.shape[0]} | Test: {X1_test.shape[0]}")
print(f"df2 → Train: {X2_train.shape[0]} | Test: {X2_test.shape[0]}")


In [ ]:
#The block handles the mssing Values
def fix_missing(df_encoded):
    # Fill numeric columns with median
    df_encoded = df_encoded.copy()
    for col in df_encoded.columns:
        if col != 'Profitability':
            median_val = df_encoded[col].median()
            df_encoded[col] = df_encoded[col].fillna(median_val)
    return df_encoded

df1_encoded = fix_missing(df1_encoded)
df2_encoded = fix_missing(df2_encoded)


# Confirm no missing values remain
print("Missing values handled")
print(f"\ndf1 missing values: {df1_encoded.isnull().sum().sum()}")
print(f"df2 missing values: {df2_encoded.isnull().sum().sum()}")


In [ ]:
#Re-split Features and Target After Missing Value Fix

#df1
X1 = df1_encoded[df1_features]
y1 = df1_encoded['Profitability']

#df2
X2 = df2_encoded[df2_features]
y2 = df2_encoded['Profitability']



# Re-do train/test split
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)



print("Re-split complete — no missing values in any split")
print(f"\ndf1 → Train: {X1_train.shape[0]} | Test: {X1_test.shape[0]}")
print(f"df2 → Train: {X2_train.shape[0]} | Test: {X2_test.shape[0]}")



In [ ]:
#Training Random Forest
# Crucial Parameters:
# n_estimators=100 -> This builds 100 decision trees
# max_depth=10 -> limit tree depth to avoid overfitting
# random_state=42 -> reproducible results every run
# class_weight='balanced' -> handles class imbalance fairly

from sklearn.ensemble import RandomForestClassifier
import time

# --- df1 ---
print("Training Random Forest on df1..")
start = time.time()
rf1 = RandomForestClassifier(n_estimators=100, max_depth=10,
                             random_state=42, class_weight='balanced')
rf1.fit(X1_train, y1_train)
t1 = time.time() - start
print(f"Training time: {t1:.2f} seconds")


# --- df2 ---
print(" Training Random Forest on df2...")
start = time.time()
rf2 = RandomForestClassifier(n_estimators=100, max_depth=10,
                              random_state=42, class_weight='balanced')
rf2.fit(X2_train, y2_train)
t2 = time.time() - start
print(f"Training time: {t2:.1f} seconds")



print(f"All 2 Random Forest models trained!")
print(f" df1: {t1:.1f}s | df2: {t2:.1f}s")


In [ ]:
#Making the predictions on Test Sets
# The Model predicts Low / Medium / High for unseen data

# Generate predictions for all 3 datasets
y1_pred = rf1.predict(X1_test)
y2_pred = rf2.predict(X2_test)


print("Predictions made for all 3 datasets")
print(f"\nSample predictions from df1:")
print(f"  Actual    : {list(y1_test[:10])}")
print(f"  Predicted : {list(y1_pred[:10])}")



In [ ]:
# Evaluation Metrics
# Accuracy  = overall correct predictions
# Precision = when model says High, how often is it right?
# Recall    = of all actual High games, how many were caught?


from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, classification_report)

def evaluate_model(name, y_test, y_pred):
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred,
                           average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred,
                        average='weighted', zero_division=0)

    print(f"\n{'='*50}")
    print(f"{name} — Random Forest Evaluation")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.1f}%)")
    print(f"  Precision : {prec:.4f} ({prec*100:.1f}%)")
    print(f"  Recall    : {rec:.4f} ({rec*100:.1f}%)")
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred,
                                 target_names=['High', 'Low', 'Medium'],
                                 zero_division=0))

evaluate_model("df1 (vgData2024)", y1_test, y1_pred)
evaluate_model("df2 (vgData2026)", y2_test, y2_pred)


In [ ]:
# CELL 18: Confusion Matrices
# Shows exactly where the model gets confused
# Rows = Actual class | Columns = Predicted class


from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
classes = ['High', 'Low', 'Medium']

for ax, (name, y_test, y_pred) in zip(axes, [
    ("df1 (vgData2024)", y1_test, y1_pred),
    ("df2 (vgData2026)", y2_test, y2_pred)
]):
    cm = confusion_matrix(y_test, y_pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=ax, cbar=False)
    ax.set_title(f'{name}\nConfusion Matrix',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('Actual Label', fontsize=10)

plt.suptitle('Random Forest — Confusion Matrices Across Datasets',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()



In [ ]:
# Feature Importaance
# Which features contributed most to the predictions
# Higher Value = more important to the model

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (name, model, features) in zip(axes, [
    ("df1 (vgData2024)", rf1, df1_features),
    ("df2 (vgData2026)", rf2, df2_features)
]):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    sorted_features = [features[i] for i in indices]
    sorted_importance = importances[indices]

    bars = ax.barh(sorted_features[::-1],
                   sorted_importance[::-1],
                   color='steelblue', edgecolor='black')
    ax.set_title(f'{name}\nFeature Importance',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=10)
    ax.set_xlim(0, max(sorted_importance) + 0.05)

    # Add value labels on bars
    for bar, val in zip(bars, sorted_importance[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() +
                bar.get_height()/2, f'{val:.3f}',
                va='center', fontsize=9)

plt.suptitle('Random Forest — Feature Importance Across Datasets',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print importance table
print("\n Feature Importance Rankings:")
for name, model, features in [
    ("df1", rf1, df1_features),
    ("df2", rf2, df2_features)
]:
    print(f"\n{name}:")
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    for rank, i in enumerate(indices, 1):
        print(f"  {rank}. {features[i]:<15} {importances[i]:.4f}")

In [ ]:
# Performance comparison acress Datasets

metrics = {
    'Dataset': ['df1 (vgData2024)', 'df2 (vgData2026)'],
    'Accuracy':  [0.9796, 0.9832],
    'Precision': [0.9795, 0.9832],
    'Recall':    [0.9796, 0.9832]
}

df_metrics = pd.DataFrame(metrics)

# Plot
x = np.arange(len(metrics['Dataset']))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, df_metrics['Accuracy'],
               width, label='Accuracy',  color='steelblue')
bars2 = ax.bar(x,         df_metrics['Precision'],
               width, label='Precision', color='seagreen')
bars3 = ax.bar(x + width, df_metrics['Recall'],
               width, label='Recall',    color='coral')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Random Forest — Performance Comparison Across Datasets',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics['Dataset'], fontsize=11)
ax.set_ylim(0.95, 1.00)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.001,
                f'{bar.get_height():.1%}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n Final Comparison Table:")
print(df_metrics.to_string(index=False))